In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain.agents import create_agent
from langchain_community.agent_toolkits import create_sql_agent



/tmp/ipykernel_7831/2455014558.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [ ]:

load_dotenv()

# Avoid hardcoding API keys in your script.
api_key = os.getenv("GEMINI_API_KEY")


groq_api_key = os.getenv("GROQ_API_KEY")

# # 1. Initialize the modern Gemini model wrapper
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite", 
#     google_api_key=api_key,
#     temperature=0.2
# )

llm = ChatGroq(
    model="llama-3.3-70b-versatile", # High-reasoning 70B model
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0 # CRITICAL: keep at 0 for deterministic SQL generation
)

# # 2. Invoke the model (LangChain uses .invoke() instead of calling the object directly)
response = llm.invoke("Write a poem on my love for dosa")

# # 3. Print the text content of the message response
print(response.content)

In [3]:
# =====================================================================
# DATABASE CONNECTION (CRITICAL SECURITY STEP: USE A READ-ONLY USER)
# =====================================================================

db_user = "root"
db_password = os.getenv("DB_PASSWORD")  # Replace with your actual password or use environment variables for better security
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)

print(db.table_info)

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '3142226@localhost' ([Errno -2] Name or service not known)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# =====================================================================
# TOOLKIT SETUP
# =====================================================================
# This automatically creates tools like:
# - sql_db_list_tables (lists visible tables)
# - sql_db_schema (fetches columns/types for a table safely)
# - sql_db_query_checker (uses LLM to inspect code for syntax errors before running)
# - sql_db_query (executes read-only commands)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [ ]:
# =====================================================================
# AGENT CREATION
# =====================================================================
agent_executor = create_agent(
    model=llm,
    tools=tools,
)

In [23]:
# =====================================================================
# EXECUTION EXAMPLES
# =====================================================================
# Safe Query
safe_prompt = "How many t-shirts do we have left for nike in extra small size and white color?"
response = agent_executor.invoke({"messages": [("user", safe_prompt)]})

# 1. Look through the agent's step-by-step message history
for msg in response["messages"]:
    # Check if the model made an internal tool call (like running a query)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tool_call in msg.tool_calls:
            if tool_call["name"] == "sql_db_query":
                print("\n===== EXACT SQL QUERY GENERATED =====")
                print(tool_call["args"]["query"])
                print("=====================================\n")

# 2. Print your final answer as usual
print("Answer:", response["messages"][-1].content)


===== EXACT SQL QUERY GENERATED =====
SELECT COUNT(*) FROM tshirts WHERE brand = 'nike' AND size = 'extra small' AND color = 'white'


===== EXACT SQL QUERY GENERATED =====
SELECT COUNT(*) FROM nike_tshirts WHERE size = 'extra small' AND color = 'white'


===== EXACT SQL QUERY GENERATED =====
SELECT COUNT(*) FROM t_shirts WHERE brand = 'nike' AND size = 'extra small' AND color = 'white'

Answer: There are 0 t-shirts left for Nike in extra small size and white color.


In [ ]:
safe_prompt = "How much is the price of the inventory for all small size t-shirts?"

response = agent_executor.invoke({"messages": [("user", safe_prompt)]})

# 1. Look through the agent's step-by-step message history
for msg in response["messages"]:
    # Check if the model made an internal tool call (like running a query)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tool_call in msg.tool_calls:
            if tool_call["name"] == "sql_db_query":
                print("\n===== EXACT SQL QUERY GENERATED =====")
                print(tool_call["args"]["query"])
                print("=====================================\n")

# 2. Print your final answer as usual
print("Answer:", response["messages"][-1].content)